In [5]:
import os
from pathlib import Path

root_is_cwd = os.getcwd().endswith("fusionLearning")

if not root_is_cwd:
    os.chdir(Path().resolve().parent.parent)
    print("Changed to root directory")
    root_is_cwd = True
else:
    print("Already in root directory")

print(os.getcwd())

Changed to root directory
C:\Users\GAMER01\codeproj\fusionLearning


In [6]:

import torch
import segmentation_models_pytorch as smp 
import matplotlib.pyplot as plt
from baseModels.dataloaders import create_train_val_test_loaders

import albumentations as A
from albumentations.pytorch import ToTensorV2
import pytorch_lightning as pl

from torch.optim import lr_scheduler


MAXEPOCHS = 50
BATCHSIZE = 1
MOMENTUM = 0.99
LEARNING_RATE = 0.01
NUM_CLASSES = 2



c:\Users\GAMER01\codeproj\fusionLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
unet = smp.Unet(
    encoder_name="resnet34",  
    encoder_weights=None,  
    in_channels=3,  
    classes=NUM_CLASSES,
)
unet.to(device)

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [9]:
path_images_folder = os.path.join("CUBdata/CUB_200_2011/images")
path_segmentations_folder = os.path.join("CUBdata/segmentations")

training_dataloader, validation_dataloader, testing_dataloader = create_train_val_test_loaders(
    image_dir=path_images_folder, 
    segmentation_dir=path_segmentations_folder,
    batch_size=BATCHSIZE
)
    

Dataset loaded with 11788 image-segmentation pairs


TypeError: pic should be PIL Image or ndarray. Got <class 'torch.Tensor'>

In [2]:
# optimizer

from torch import optim


optimizer = torch.optim.SGD(unet.parameters(),
                           lr=LEARNING_RATE,
                           momentum=MOMENTUM)

# loss function
lossFunc = torch.nn.CrossEntropyLoss()

unet.compile()



NameError: name 'torch' is not defined

In [ ]:
trainer = pl.Trainer(max_epochs=MAXEPOCHS, gpus=1 if torch.cuda.is_available() else 0,
                     callbacks=[
                        pl.callbacks.ModelCheckpoint(
                            monitor="val_loss",
                            dirpath="outputs",
                            filename="best_model",
                            save_top_k=1,
                            mode="min"
                        )
                     ],
                     
                     )

trainer.fit(model=unet, train_dataloaders=training_dataloader, 
            val_dataloaders=validation_dataloader, 
            lr=LEARNING_RATE, momentum=MOMENTUM
)

trainer.test(model=unet, dataloaders=testing_dataloader)

trainer.save_checkpoint("outputs/best_model.pth")



